# Qwen3-4B-Instruct-2507 — English↔Mandarin Chinese LoRA Fine-tuning

This notebook fine-tunes `Qwen/Qwen3-4B-Instruct-2507` on English↔Mandarin Chinese translation using **Unsloth** with QLoRA on a single T4 GPU (16 GB VRAM). Two LoRA adapters are trained:
1. **EN→ZH**: English to Mandarin Chinese
2. **ZH→EN**: Mandarin Chinese to English

The training data comes from Tatoeba (~32k sentence pairs). A separate eval set is used for `eval_loss` tracking and best-checkpoint selection.

> **Google Colab**: Run on a GPU runtime (Runtime → Change runtime type → T4 GPU).

## 1. Setup and Dependencies

Install Unsloth with pinned dependency versions, mount Google Drive, and verify GPU availability.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --force-reinstall --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --no-deps "trl" "peft" "accelerate"
!pip install datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ex1oki7j/unsloth_8dea3183ef4844e894e1dc6524fe1ea6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ex1oki7j/unsloth_8dea3183ef4844e894e1dc6524fe1ea6
  Resolved https://github.com/unslothai/unsloth.git to commit b3640802253f64117ee228718be7fab32e47aa5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-install-73ii2s9a/unsloth-zoo_74c23a735ab44db985a3f8bf2ab4d816
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth-zoo.git /tmp/pip-install-73ii2s9a/unsloth-zoo_74c23a735ab44db985a3f8bf2ab4d816
  Resolved https://github.com/unslothai/unsloth-zoo.git to commit 232d950935cd4812f4dd5a61b0208192afe75891
  Installing build dependencies ... done
  Getting requirements to buil

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

# === Configuration constants ===
PROJECT_DIR = "/content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-ft"
REVERSE_PROJECT_DIR = "/content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-en-ft"
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 512
SYSTEM_PROMPT = ("You are a professional translator. Translate English to "
                 "Mandarin Chinese. Output only the translation, no explanation.")

# Data: reuse the same Tatoeba file from the Llama project
DATA_PATH = "/content/drive/MyDrive/Coding project/Llama_Translations/llama32-zh-ft/cmn.txt"

# Create output subdirectories for both adapters
for proj_dir in [PROJECT_DIR, REVERSE_PROJECT_DIR]:
    for subdir in ["final_adapter", "checkpoints", "logs"]:
        os.makedirs(f"{proj_dir}/{subdir}", exist_ok=True)

print(f"EN→ZH output: {PROJECT_DIR}")
print(f"ZH→EN output: {REVERSE_PROJECT_DIR}")
print(f"Data source: {DATA_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EN→ZH output: /content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-ft
ZH→EN output: /content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-en-ft
Data source: /content/drive/MyDrive/Coding project/Llama_Translations/llama32-zh-ft/cmn.txt


In [3]:
!nvidia-smi

import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
gpu_name = result.stdout.strip()
assert "T4" in gpu_name, f"Expected T4 GPU, got: {gpu_name}"
print(f"Running on {gpu_name}")

Sun May 10 21:26:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Load Model and Tokenizer with Unsloth

Load Qwen3-4B-Instruct-2507 in 4-bit quantization (QLoRA) using Unsloth's optimized loader. No HuggingFace login is required — this model is publicly accessible.

In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,            # auto: fp16 on T4
    load_in_4bit=True,     # QLoRA
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Chat template: ChatML (im_start/im_end)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: Qwen/Qwen3-4B-Instruct-2507
Chat template: ChatML (im_start/im_end)


## 3. Attach LoRA Adapter

Attach a LoRA adapter targeting all attention and MLP projection layers. Rank 16 with alpha 16. Dropout is 0 to enable Unsloth's optimized fast path.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,        # 0 enables Unsloth's fast path
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 4. Load and Format Datasets

Load the Tatoeba sentence pairs from the shared `cmn.txt` file (tab-separated). Format each example using Qwen's ChatML chat template with a system prompt instructing translation behavior.

In [6]:
import pandas as pd
from datasets import Dataset

# Load data from shared location
train_df = pd.read_csv(DATA_PATH, sep="\t", header=None,
                       names=["en", "zh", "attribution"], quoting=3)

# Clean
train_df.drop(columns=["attribution"], inplace=True)
train_df.dropna(inplace=True)
train_df.drop_duplicates(subset=["en", "zh"], inplace=True)

# TEST MODE: only use 20 sentences for a quick sanity check
TEST_MODE = False
if TEST_MODE:
    train_df = train_df.head(20)

print(f"Total examples: {len(train_df)}")

# Split 5% for validation (used for early stopping)
val_df = train_df.sample(frac=0.05, random_state=42)
train_df = train_df.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")

# Format with chat template (Qwen ChatML format)
def format_example(row):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": row["en"]},
        {"role": "assistant", "content": row["zh"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
train_ds = train_ds.map(format_example, remove_columns=["en", "zh"])

val_ds = Dataset.from_pandas(val_df, preserve_index=False)
val_ds = val_ds.map(format_example, remove_columns=["en", "zh"])

print(f"\nTraining examples: {len(train_ds):,}")
print(f"Validation examples: {len(val_ds):,}")
print("\nSample formatted text:")
print(train_ds[0]["text"])

Total examples: 32028
Train: 30,427 | Val: 1,601


Map:   0%|          | 0/30427 [00:00<?, ? examples/s]

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]


Training examples: 30,427
Validation examples: 1,601

Sample formatted text:
<|im_start|>system
You are a professional translator. Translate English to Mandarin Chinese. Output only the translation, no explanation.<|im_end|>
<|im_start|>user
Hi.<|im_end|>
<|im_start|>assistant
<think>

</think>

嗨。<|im_end|>



## 5. Configure SFTTrainer

Effective batch size of 16 (8 × 2 gradient accumulation), cosine learning rate schedule, save every 200 steps. Checkpoints saved locally then rsynced to Drive.

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback, EarlyStoppingCallback

class DriveSyncCallback(TrainerCallback):
    """Save to local disk, then rsync to Drive."""
    def on_save(self, args, state, control, **kwargs):
        os.system(f"rsync -a /content/checkpoints/ {PROJECT_DIR}/checkpoints/")

training_args = TrainingArguments(
    output_dir="/content/checkpoints",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    logging_dir=f"{PROJECT_DIR}/logs",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
    callbacks=[DriveSyncCallback(), EarlyStoppingCallback(early_stopping_patience=3)],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/30427 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1601 [00:00<?, ? examples/s]

## 6. Train on Completions Only

Mask the prompt (system + user turns) from the loss so the model only learns to generate the translation. The response marker for Qwen's ChatML format is `<|im_start|>assistant\n`.

In [8]:
RESPONSE_MARKER = "<|im_start|>assistant\n"
response_token_ids = tokenizer.encode(RESPONSE_MARKER, add_special_tokens=False)

trainer.train_dataset.reset_format()
trainer.eval_dataset.reset_format()

def mask_prompt(example):
    input_ids = list(example["input_ids"])
    labels = list(input_ids)

    resp_len = len(response_token_ids)
    response_start = None
    for i in range(len(input_ids) - resp_len + 1):
        if input_ids[i:i + resp_len] == response_token_ids:
            response_start = i + resp_len
            break

    if response_start is not None:
        labels[:response_start] = [-100] * response_start
    else:
        labels = [-100] * len(labels)

    example["labels"] = labels
    return example

trainer.train_dataset = trainer.train_dataset.map(mask_prompt)
trainer.eval_dataset = trainer.eval_dataset.map(mask_prompt)

sample = trainer.train_dataset[0]
input_ids = sample["input_ids"][:20]
labels = sample["labels"][:20]
print("First 20 tokens:", tokenizer.decode(input_ids))
print("First 20 labels:", labels)
print("(Labels = -100 means that token is masked from loss)")

Map:   0%|          | 0/30427 [00:00<?, ? examples/s]

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

First 20 tokens: <|im_start|>system
You are a professional translator. Translate English to Mandarin Chinese. Output only the translation,
First 20 labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
(Labels = -100 means that token is masked from loss)


## 7. Launch TensorBoard and Train

In [9]:
os.environ["TB_LOGS"] = f"{PROJECT_DIR}/logs"

%load_ext tensorboard
%tensorboard --logdir "$TB_LOGS"

<IPython.core.display.Javascript object>

In [10]:
import glob

ckpts = sorted(glob.glob(f"{PROJECT_DIR}/checkpoints/checkpoint-*"))
resume = ckpts[-1] if ckpts else None
if resume:
    print(f"Resuming from checkpoint: {resume}")

trainer_stats = trainer.train(resume_from_checkpoint=resume)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30,427 | Num Epochs = 2 | Total steps = 3,804
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
200,0.906631,0.941363
400,0.935533,0.913356
600,0.873368,0.899491
800,0.878118,0.894817
1000,0.881928,0.886148
1200,0.882690,0.883876
1400,0.887815,0.877218
1600,0.885909,0.873006
1800,0.832316,0.872330
2000,0.834895,0.870948


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

## 8. Save EN→ZH Adapter

In [11]:
model.save_pretrained(f"{PROJECT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{PROJECT_DIR}/final_adapter")
print(f"Adapter saved to {PROJECT_DIR}/final_adapter")

peak_memory_gb = torch.cuda.max_memory_reserved() / 1e9
total_time_min = trainer_stats.metrics["train_runtime"] / 60
train_loss = trainer_stats.metrics["train_loss"]

print(f"\n{'='*50}")
print(f"EN->ZH Training complete!")
print(f"  Total runtime:   {total_time_min:.1f} minutes")
print(f"  Peak VRAM:       {peak_memory_gb:.2f} GB")
print(f"  Final train loss: {train_loss:.4f}")
print(f"{'='*50}")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-ft/final_adapter/tokenizer_config.json.


Adapter saved to /content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-ft/final_adapter

EN->ZH Training complete!
  Total runtime:   107.9 minutes
  Peak VRAM:       8.93 GB
  Final train loss: 0.9260


## 9. Train Reverse Adapter (Chinese → English)

Train a second LoRA adapter for the reverse direction (ZH→EN) using the same Tatoeba data with swapped source/target.

In [12]:
import gc

del model
del trainer
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed — reloading base model for reverse adapter")

GPU memory freed — reloading base model for reverse adapter


In [13]:
REVERSE_SYSTEM_PROMPT = ("You are a professional translator. Translate Mandarin Chinese to "
                         "English. Output only the translation, no explanation.")

# Reload base model
rev_model, rev_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

rev_model = FastLanguageModel.get_peft_model(
    rev_model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("Base model reloaded with new LoRA adapter for ZH->EN")

==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Base model reloaded with new LoRA adapter for ZH->EN


In [14]:
# Reuse the same split data but swap directions: user=zh, assistant=en
def format_reverse_example(row):
    messages = [
        {"role": "system",    "content": REVERSE_SYSTEM_PROMPT},
        {"role": "user",      "content": row["zh"]},
        {"role": "assistant", "content": row["en"]},
    ]
    return {"text": rev_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)}

rev_train_ds = Dataset.from_pandas(train_df, preserve_index=False)
rev_train_ds = rev_train_ds.map(format_reverse_example, remove_columns=["en", "zh"])

rev_val_ds = Dataset.from_pandas(val_df, preserve_index=False)
rev_val_ds = rev_val_ds.map(format_reverse_example, remove_columns=["en", "zh"])

print(f"Reverse training examples: {len(rev_train_ds):,}")
print(f"Reverse validation examples: {len(rev_val_ds):,}")
print("\nSample:")
print(rev_train_ds[0]["text"])

Map:   0%|          | 0/30427 [00:00<?, ? examples/s]

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

Reverse training examples: 30,427
Reverse validation examples: 1,601

Sample:
<|im_start|>system
You are a professional translator. Translate Mandarin Chinese to English. Output only the translation, no explanation.<|im_end|>
<|im_start|>user
嗨。<|im_end|>
<|im_start|>assistant
<think>

</think>

Hi.<|im_end|>



In [15]:
class ReverseDriveSyncCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        os.system(f"rsync -a /content/checkpoints_rev/ {REVERSE_PROJECT_DIR}/checkpoints/")

rev_training_args = TrainingArguments(
    output_dir="/content/checkpoints_rev",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    logging_dir=f"{REVERSE_PROJECT_DIR}/logs",
    seed=42,
)

rev_trainer = SFTTrainer(
    model=rev_model,
    processing_class=rev_tokenizer,
    train_dataset=rev_train_ds,
    eval_dataset=rev_val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=rev_training_args,
    callbacks=[ReverseDriveSyncCallback(), EarlyStoppingCallback(early_stopping_patience=3)],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/30427 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1601 [00:00<?, ? examples/s]

In [16]:
RESPONSE_MARKER = "<|im_start|>assistant\n"
rev_response_token_ids = rev_tokenizer.encode(RESPONSE_MARKER, add_special_tokens=False)

rev_trainer.train_dataset.reset_format()
rev_trainer.eval_dataset.reset_format()

def mask_prompt_rev(example):
    input_ids = list(example["input_ids"])
    labels = list(input_ids)

    resp_len = len(rev_response_token_ids)
    response_start = None
    for i in range(len(input_ids) - resp_len + 1):
        if input_ids[i:i + resp_len] == rev_response_token_ids:
            response_start = i + resp_len
            break

    if response_start is not None:
        labels[:response_start] = [-100] * response_start
    else:
        labels = [-100] * len(labels)

    example["labels"] = labels
    return example

rev_trainer.train_dataset = rev_trainer.train_dataset.map(mask_prompt_rev)
rev_trainer.eval_dataset = rev_trainer.eval_dataset.map(mask_prompt_rev)

sample = rev_trainer.train_dataset[0]
print("First 20 tokens:", rev_tokenizer.decode(sample["input_ids"][:20]))
print("First 20 labels:", sample["labels"][:20])

Map:   0%|          | 0/30427 [00:00<?, ? examples/s]

Map:   0%|          | 0/1601 [00:00<?, ? examples/s]

First 20 tokens: <|im_start|>system
You are a professional translator. Translate Mandarin Chinese to English. Output only the translation,
First 20 labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [17]:
import glob

rev_ckpts = sorted(glob.glob(f"{REVERSE_PROJECT_DIR}/checkpoints/checkpoint-*"))
rev_resume = rev_ckpts[-1] if rev_ckpts else None
if rev_resume:
    print(f"Resuming from checkpoint: {rev_resume}")

rev_trainer_stats = rev_trainer.train(resume_from_checkpoint=rev_resume)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30,427 | Num Epochs = 2 | Total steps = 3,804
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss,Validation Loss
200,0.662912,0.655493
400,0.626081,0.633598
600,0.625413,0.625189
800,0.609017,0.622418
1000,0.616094,0.617971
1200,0.596673,0.616994
1400,0.601154,0.611371
1600,0.615502,0.610199
1800,0.568402,0.609993
2000,0.597988,0.609743


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

In [18]:
rev_model.save_pretrained(f"{REVERSE_PROJECT_DIR}/final_adapter")
rev_tokenizer.save_pretrained(f"{REVERSE_PROJECT_DIR}/final_adapter")
print(f"Adapter saved to {REVERSE_PROJECT_DIR}/final_adapter")

peak_memory_gb = torch.cuda.max_memory_reserved() / 1e9
total_time_min = rev_trainer_stats.metrics["train_runtime"] / 60
train_loss = rev_trainer_stats.metrics["train_loss"]

print(f"\n{'='*50}")
print(f"ZH->EN Reverse adapter training complete!")
print(f"  Total runtime:   {total_time_min:.1f} minutes")
print(f"  Peak VRAM:       {peak_memory_gb:.2f} GB")
print(f"  Final train loss: {train_loss:.4f}")
print(f"{'='*50}")
print(f"\nBoth adapters saved:")
print(f"  EN->ZH: {PROJECT_DIR}/final_adapter/")
print(f"  ZH->EN: {REVERSE_PROJECT_DIR}/final_adapter/")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-en-ft/final_adapter/tokenizer_config.json.


Adapter saved to /content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-en-ft/final_adapter

ZH->EN Reverse adapter training complete!
  Total runtime:   107.0 minutes
  Peak VRAM:       9.71 GB
  Final train loss: 0.6677

Both adapters saved:
  EN->ZH: /content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-ft/final_adapter/
  ZH->EN: /content/drive/MyDrive/Coding project/Qwen_Translations/qwen3-zh-en-ft/final_adapter/
